# 01 — Data Ingestion & Quality Assessment

Role: determine whether raw 2025 Yellow Taxi data is reliable before any downstream
analysis.

Scope: load 12 months, audit schema, missing values, duplicates, invalid records, date
integrity, duration sanity, outlier bounds, and payment-code distribution.

Output: a documented cleaning specification, implemented in `preprocessing.clean_trips`.

Year check: YEAR = 2025 loaded successfully across all 12 files.

In [1]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np

from data_loader import load_month
from quality_checks import (
    check_missing, check_duplicates, check_value_ranges,
    check_outliers_iqr, check_class_balance, run_quality_report
)

pd.set_option('display.max_columns', None)
YEAR = 2025
MONTHS = range(1, 13)

## 1. Load All 12 Months (Raw)
Loads each month independently so a single bad file fails loudly rather than
silently.

In [2]:
raw_frames = {}
load_errors = {}

for m in MONTHS:
    try:
        raw_frames[m] = load_month(YEAR, m)
        print(f"month={m:02d} rows={len(raw_frames[m]):,}")
    except Exception as e:
        load_errors[m] = str(e)
        print(f"month={m:02d} FAILED: {e}")

print(f"\nloaded={len(raw_frames)}/12 months, failed={len(load_errors)}")

month=01 rows=3,475,226
month=02 rows=3,577,543
month=03 rows=4,145,257
month=04 rows=3,970,553
month=05 rows=4,591,845
month=06 rows=4,322,960
month=07 rows=3,898,963
month=08 rows=3,574,091
month=09 rows=4,251,015
month=10 rows=4,428,699
month=11 rows=4,181,444
month=12 rows=4,305,006

loaded=12/12 months, failed=0


**Results:** 12 of 12 months loaded successfully, with no failures. Row counts range
from 3.48 million (January) to 4.59 million (May), for a full-year raw total of
approximately 48.7 million rows.

## 2. Schema Consistency Across Months
Confirms every month shares identical columns and data types before the pipeline
assumes that consistency.

In [3]:
schemas = {m: tuple(sorted(df.columns)) for m, df in raw_frames.items()}
unique_schemas = set(schemas.values())

print(f"distinct schemas across months: {len(unique_schemas)}")
if len(unique_schemas) > 1:
    for m, s in schemas.items():
        print(f"month={m:02d} schema_hash={hash(s)}")
else:
    print("all 12 months share identical column set.")

raw_frames[1].dtypes

distinct schemas across months: 1
all 12 months share identical column set.


VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
data_year                         int64
data_month                        int64
dtype: object

In [4]:
dtype_ref = raw_frames[1].dtypes
dtype_drift = []
for m, df in raw_frames.items():
    diff = df.dtypes[df.dtypes != dtype_ref.reindex(df.dtypes.index)]
    if not diff.empty:
        dtype_drift.append((m, diff))

if dtype_drift:
    for m, diff in dtype_drift:
        print(f"month={m:02d} dtype drift:\n{diff}\n")
else:
    print("no dtype drift vs month 1 reference.")

no dtype drift vs month 1 reference.


**Results:** all 12 months share a single, identical schema, with no data type drift
relative to the January reference. The schema is stable, so a single pipeline can be
applied across the full year without per-month adjustments.

## 3. Row Counts by Month
Baseline monthly volume, used as the reference point for retention-rate checks after
cleaning in later notebooks.

In [5]:
row_counts = pd.DataFrame({
    'month': list(raw_frames.keys()),
    'raw_rows': [len(df) for df in raw_frames.values()],
}).sort_values('month')
row_counts

,month,raw_rows
0,1,3475226
1,2,3577543
2,3,4145257
3,4,3970553
4,5,4591845
5,6,4322960
6,7,3898963
7,8,3574091
8,9,4251015
9,10,4428699


**Results:** volume is lowest in January and August (approximately 3.5 million rows)
and highest in May (4.59 million rows). No month is anomalously low at the raw
ingestion stage; any zero-row failure identified later belongs to the cleaning step,
not ingestion.

## 4. Missing Values
Reports per-month missing count and percentage for every column.

In [6]:
missing_by_month = {}
for m, df in raw_frames.items():
    miss = check_missing(df)
    missing_by_month[m] = miss
    if not miss.empty:
        print(f"--- month={m:02d} ---")
        print(miss)
    else:
        print(f"month={m:02d}: no missing values")

--- month=01 ---
                      missing_count  missing_pct
passenger_count              540149        15.54
RatecodeID                   540149        15.54
store_and_fwd_flag           540149        15.54
congestion_surcharge         540149        15.54
Airport_fee                  540149        15.54
--- month=02 ---
                      missing_count  missing_pct
passenger_count              806937        22.56
RatecodeID                   806937        22.56
store_and_fwd_flag           806937        22.56
congestion_surcharge         806937        22.56
Airport_fee                  806937        22.56
--- month=03 ---
                      missing_count  missing_pct
passenger_count              916663        22.11
RatecodeID                   916663        22.11
store_and_fwd_flag           916663        22.11
congestion_surcharge         916663        22.11
Airport_fee                  916663        22.11
--- month=04 ---
                      missing_count  missing_pct
p

**Results:** five columns — `passenger_count`, `RatecodeID`, `store_and_fwd_flag`,
`congestion_surcharge`, and `Airport_fee` — are missing together, with an identical row
count each month. This indicates one shared root cause rather than five independent
issues, consistent with a trip-record subtype that does not populate this column group.
The missing rate ranges from 15.5% in January to 28.1% in June, with no clear seasonal
pattern. This missingness is the direct source of the `passenger_count_invalid` volume
reported in Section 6.

## 5. Duplicate Records
Checks for exact-row duplicates across all 12 months.

In [7]:
dup_rows = []
for m, df in raw_frames.items():
    d = check_duplicates(df)
    d['month'] = m
    dup_rows.append(d)

dup_summary = pd.DataFrame(dup_rows)[['month', 'duplicate_rows', 'duplicate_pct']]
dup_summary

,month,duplicate_rows,duplicate_pct
0,1,0,0.0
1,2,0,0.0
2,3,0,0.0
3,4,0,0.0
4,5,0,0.0
5,6,0,0.0
6,7,1,0.0
7,8,0,0.0
8,9,0,0.0
9,10,0,0.0


**Results:** duplication is effectively absent — one duplicate row across the full
year (July), representing 0.00% of every month. This is not a cleaning concern.

## 6. Invalid / Non-Physical Value Checks
Flags negative or zero fare, total amount, and distance, along with out-of-range
`passenger_count`, on the raw pre-cleaning data.

In [8]:
range_rows = []
for m, df in raw_frames.items():
    vr = check_value_ranges(df)
    vr['month'] = m
    range_rows.append(vr)

range_summary = pd.concat(range_rows, ignore_index=True)
range_summary.pivot(index='rule', columns='month', values='violation_count')

month,1,2,3,4,5,6,7,8,9,10,11,12
rule,,,,,,,,,,,,
fare_amount_non_positive,145516,184096,210378,188171,327375,277527,248458,262635,254424,323383,397555,50716
passenger_count_invalid,564823,828704,939391,768845,1220155,1235748,1059320,903515,1089384,1013210,1034691,1214316
total_amount_non_positive,63596,55621,69287,73222,113064,74888,77038,88458,79109,109312,127437,49490
trip_distance_non_positive,90893,99771,103722,91439,141121,134896,123774,105184,124483,125599,109420,152656


**Results:** `passenger_count_invalid` accounts for the large majority of flagged
rows every month (569,000 to 1.24 million), and tracks the missing-value pattern
identified in Section 4, since a missing value fails the valid 1–6 range check.
Non-positive `fare_amount`, `total_amount`, and `trip_distance` violations are smaller
by comparison, at 5–18% of the passenger_count figure, and trend upward through the
year alongside overall trip volume.

## 7. Date Range Integrity
Checks that pickup timestamps fall within the file's stated year and month.

In [9]:
date_issues = []
for m, df in raw_frames.items():
    pu = pd.to_datetime(df['tpep_pickup_datetime'])
    outside = ((pu.dt.year != YEAR) | (pu.dt.month != m)).sum()
    date_issues.append({
        'month': m,
        'outside_stated_month': int(outside),
        'outside_pct': round(100 * outside / len(df), 3),
        'min_pickup': pu.min(),
        'max_pickup': pu.max(),
    })

pd.DataFrame(date_issues)

,month,outside_stated_month,outside_pct,min_pickup,max_pickup
0,1,22,0.001,2024-12-31 20:47:55,2025-02-01 00:00:44
1,2,31,0.001,2025-01-31 22:22:53,2025-03-01 00:06:32
2,3,33,0.001,2007-12-05 18:45:00,2025-04-01 00:00:17
3,4,7,0.000,2025-03-31 23:45:01,2025-05-01 00:48:13
4,5,24,0.001,2009-01-01 00:20:39,2025-06-01 00:04:31
5,6,20,0.000,2025-05-31 22:34:26,2025-06-30 23:59:59
6,7,7,0.000,2009-01-01 08:52:26,2025-07-31 23:59:59
7,8,17,0.000,2009-01-01 12:52:15,2025-09-01 00:00:29
8,9,7,0.000,2025-08-31 23:45:38,2025-10-01 00:00:11
9,10,13,0.000,2025-09-30 22:54:51,2025-11-01 00:32:12


**Results:** volume outside the stated month is negligible, at 7 to 33 rows per
month (0.001% or less). This is not limited to month-boundary spillover: several
months contain pickup timestamps from 2007, 2008, and 2009, appearing in March, May,
July, August, and November. The row count is small, but this reflects genuine data
corruption rather than a rounding effect at month boundaries.

## 8. Duration Sanity
Confirms that dropoff time is strictly later than pickup time.

In [10]:
duration_issues = []
for m, df in raw_frames.items():
    pu = pd.to_datetime(df['tpep_pickup_datetime'])
    do = pd.to_datetime(df['tpep_dropoff_datetime'])
    non_positive = (do <= pu).sum()
    duration_issues.append({
        'month': m,
        'non_positive_duration': int(non_positive),
        'pct': round(100 * non_positive / len(df), 3),
    })

pd.DataFrame(duration_issues)

,month,non_positive_duration,pct
0,1,2051,0.059
1,2,5164,0.144
2,3,22280,0.537
3,4,34769,0.876
4,5,64272,1.400
5,6,68583,1.586
6,7,56064,1.438
7,8,47833,1.338
8,9,57179,1.345
9,10,67968,1.535


**Results:** non-positive duration is small in absolute terms but increases through
the year, from 0.06% in January to a peak of 1.59% in June, then settling between 1.3%
and 1.5% from July through December. From mid-year onward this represents a
consistent 57,000 to 69,000 rows per month.

## 9. Outlier Scan (IQR) — Full Year
Computes IQR bounds per month as statistical flags only; no rows are removed at this
stage.

In [11]:
columns = [
    'trip_distance',
    'fare_amount',
    'tip_amount',
    'total_amount'
]

outlier_reports = []

for name, df in raw_frames.items():
    report = check_outliers_iqr(df, columns=columns)
    report.insert(0, 'dataset', name)
    outlier_reports.append(report)

outlier_report = pd.concat(outlier_reports, ignore_index=True)

outlier_report

,dataset,column,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_pct
0,1,trip_distance,0.98,3.10,2.12,-2.200,6.280,422436,12.16
1,1,fare_amount,8.60,19.50,10.90,-7.750,35.850,381907,10.99
2,1,tip_amount,0.00,3.93,3.93,-5.895,9.825,194129,5.59
3,1,total_amount,15.20,27.78,12.58,-3.670,46.650,402661,11.59
4,2,trip_distance,1.00,3.18,2.18,-2.270,6.450,408562,11.42
5,2,fare_amount,8.60,19.80,11.20,-8.200,36.600,345405,9.65
6,2,tip_amount,0.00,3.79,3.79,-5.685,9.475,183539,5.13
7,2,total_amount,15.31,27.97,12.66,-3.680,46.960,373768,10.45
8,3,trip_distance,1.03,3.42,2.39,-2.555,7.005,489035,11.80
9,3,fare_amount,8.60,21.25,12.65,-10.375,40.225,397910,9.60


**Results:** IQR upper bounds widen through the year — `fare_amount` moves from
$35.85 in January to $55.22 in December, and `trip_distance` from 6.28 miles in January
to 9.0 miles in August. The proportion of flagged outliers stays comparatively stable
across months regardless: approximately 11–12% for `trip_distance`, 8–11% for
`fare_amount`, 9–12% for `total_amount`, and 5–6% for `tip_amount`. A stable outlier
share alongside a widening bound indicates a shift in the underlying distribution, such
as fare increases or longer trips, rather than increasing noise in the data.

## 10. Payment Type Class Balance
Reports the monthly and full-year distribution of payment codes.

In [12]:
# Monthly Analysis
balance_reports = []

for name, df in raw_frames.items():
    report = check_class_balance(df, 'payment_type')
    report.insert(0, 'month', name)
    balance_reports.append(report)

payment_balance = pd.concat(
    balance_reports,
    ignore_index=True
)

payment_balance

,month,count,pct
0,1,2444393,70.34
1,1,540149,15.54
2,1,390429,11.23
3,1,76481,2.20
4,1,23773,0.68
...,...,...,...
57,12,2618772,60.83
58,12,1195482,27.77
59,12,401019,9.32
60,12,69145,1.61


In [13]:
# Yearly Analysis
payment_counts = {}

for name, df in raw_frames.items():

    counts = df['payment_type'].value_counts()

    for payment_type, count in counts.items():
        payment_counts[payment_type] = (
            payment_counts.get(payment_type, 0) + count
        )

payment_balance = pd.DataFrame(
    list(payment_counts.items()),
    columns=['payment_type', 'count']
)

payment_balance['percentage'] = (
    payment_balance['count']
    / payment_balance['count'].sum()
    * 100
)

payment_balance = payment_balance.sort_values(
    'count',
    ascending=False
).reset_index(drop=True)

payment_balance

,payment_type,count,percentage
0,1,31054000,63.736333
1,0,11611894,23.832664
2,2,4654345,9.552743
3,4,1094213,2.245802
4,3,308147,0.632452
5,5,3,0.000006


**Results:** across the full year (48.7 million trips), payment code 1 (credit card)
accounts for 63.7%, code 2 (cash) for 9.6%, code 4 (dispute) for 2.2%, code 3 (no
charge) for 0.6%, and code 5 (unknown) for less than 0.001%. Payment code 0 accounts
for 23.8% of all trips and is not defined in the TLC data dictionary, which documents
only codes 1 through 6. Given its size, this code cannot be treated as a minor residual
category; downstream payment-method analysis should retain it as a distinct category
rather than merging it with cash or credit card.

## 11. Data Quality Summary and Cleaning Requirements

Findings:
- Schema is consistent across all 12 months, with no data type drift (Section 2).
- Missing values form a single pattern across five columns, affecting 15.5% to 28.1% of
  rows per month (Section 4).
- Duplicates are negligible, with one row across the full year (Section 5).
- Invalid values are dominated by `passenger_count_invalid` (569,000 to 1.24 million
  rows per month), driven by the missing-value pattern in Section 4; fare, total, and
  distance violations are smaller and secondary (Section 6).
- Date range integrity issues are low in volume but include genuinely corrupt
  timestamps from 2007 to 2009, not only month-boundary spillover (Section 7).
- Non-positive trip duration increases through the year, peaking at 1.59% in June
  (Section 8).
- Outlier bounds widen through the year while the outlier share remains stable,
  indicating a distribution shift rather than increasing noise (Section 9).
- Payment code 0, which is undocumented, accounts for 23.8% of all trips, the most
  significant finding in this notebook (Section 10).

Cleaning requirements, implemented in `preprocessing.clean_trips`:
1. Drop exact duplicates.
2. Drop rows missing critical fields, including datetimes, distance, fare, total
   amount, and location IDs.
3. Fill missing `passenger_count` with 1, the modal value, and cast to integer. This is
   a documented modeling assumption.
4. Drop rows where dropoff time is not strictly later than pickup time.
5. Drop rows where the pickup date falls outside the file's stated year and month.
6. Drop rows with non-positive fare, total amount, or distance.
7. Drop rows with `passenger_count` outside the range 1 to 6.
8. Outliers are not removed at the cleaning stage. They are addressed explicitly, using
   IQR and percentile thresholds, in the anomaly-detection notebook.
9. Payment code 0 is not treated as a cleaning target. It is retained and analyzed as
   its own category, as described in Section 10.

Verdict: the raw data is usable following `clean_trips` preprocessing. Two items should
be carried forward explicitly rather than resolved silently by cleaning: the
passenger_count fill-with-1 assumption in item 3, and the undocumented payment code 0,
which represents 23.8% of trips (Section 10). The next step is
`02_monthly_mobility_analysis.ipynb`.